# Experiment 04 — PhysicsFrNO Tunable Training Pipeline

This notebook provides a **complete, tunable training pipeline** for PhysicsFrNO with hyperparameter configuration based on the baseline analysis document.

**Key Features:**
- **All hyperparameters at the top** - Easy to tune without code changes
- PhysicsFrNO architecture with configurable regularization
- No auto-resume from old checkpoints (prevents architecture mismatch crashes)
- Supports both single-phase and two-phase training (log_rmse → weighted_mae)
- W&B integration for experiment tracking

## How to Tune
Edit the `HYPERPARAMS` dictionary in the Configuration cell to adjust:
- Model architecture (width, modes, depth)
- Training parameters (epochs, lr, batch size, etc.)
- Loss function settings (phase switch, intensity weighting)
- Data sampling and weighting
- Inference (TTA + autoregressive rollout)

**Pipeline:**
1. Set hyperparameters at the top
2. Resolve paths + import `src` modules
3. Load config and apply hyperparameter patch
4. Build/load scaler + bounds
5. Build train/validation dataloaders
6. Train PhysicsFrNO
7. Run inference and save `preds.npy`

In [1]:
import os, sys
import numpy as np
import torch

# ── Path resolution: local + Kaggle ─────────────────────────────────────────
KAGGLE_SRC_DATASET = "ronit-pm25-src"
KAGGLE_DATA_ROOT = "/kaggle/input/datasets/khushisingh942004/aisehack"

if os.path.exists('/kaggle'):
    os.environ['AISEHACK_DATA'] = KAGGLE_DATA_ROOT

LOCAL_SRC = os.path.abspath('../')
_kaggle_candidates = [
    f'/kaggle/input/{KAGGLE_SRC_DATASET}',
    f'/kaggle/input/datasets/ronitraj1/{KAGGLE_SRC_DATASET}',
]
KAGGLE_SRC = next(
    (p for p in _kaggle_candidates if os.path.exists(os.path.join(p, 'src'))),
    _kaggle_candidates[0],
)
SRC_ROOT = KAGGLE_SRC if os.path.exists('/kaggle') else LOCAL_SRC

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f"SRC_ROOT: {SRC_ROOT}")
print(f"AISEHACK_DATA: {os.environ.get('AISEHACK_DATA', 'not set')}")

from src.config import load_config
from src.utils import seed_everything, print_device_info, count_parameters, sanity_check_bounds
from src.data import (
    build_grid_stats,
    describe_grid_stats,
    load_grid_stats,
    load_minmax_bounds,
    load_all_months,
    get_dataloaders,
    build_test_input,
    denormalize_cpm25,
    compute_derived_features,
    compute_sparse_masks,
)
from src.model import build_model
from src.train import train, init_wandb, finish_wandb, PERSISTENCE_RMSE_PHYS
from src.inference import run_inference

SRC_ROOT: /kaggle/input/datasets/ronitraj1/ronit-pm25-src
AISEHACK_DATA: /kaggle/input/datasets/khushisingh942004/aisehack


## W&B Login (Optional but Recommended)

If `WANDB_API_KEY` is set (Kaggle Secret or environment variable), this cell logs in automatically.
If not set, training still runs but W&B syncing will be skipped if login fails.

In [2]:
import os

WANDB_API_KEY_DIRECT = "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"

try:
    import wandb
    api_key = WANDB_API_KEY_DIRECT.strip() or os.environ.get('WANDB_API_KEY', '').strip()
    if api_key:
        os.environ['WANDB_API_KEY'] = api_key
        wandb.login(key=api_key, relogin=True)
        print('W&B login complete (using direct API key).')
    else:
        print('No W&B API key found. Proceeding without explicit W&B login.')
except Exception as e:
    print(f'W&B login skipped/failed: {e}')
    print('Training can still continue without W&B syncing.')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ronitraj to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login complete (using direct API key).


## 1. Configuration

In [3]:
# =============================================================================
# HYPERPARAMETERS - EDIT THESE VALUES TO TUNE THE MODEL
# Based on baseline_tfno2d_analysis.md.resolved parameters
# =============================================================================
HYPERPARAMS = {
    # Model Architecture
    'model_type': 'physics_frno',    # changed to physics_frno
    'width': 64,                     # Hidden channel dimension (default: 64)
    'modes': 20,                     # Fourier modes retained per dimension (default: 20)
    'depth': 6,                      # Number of FNO blocks (default: 6)
    
    # Feature Configuration
    'use_aux': True,                 # Use auxiliary features (enabled for physics_frno)
    
    # Training Configuration
    'epochs': 50,                    # Total training epochs
    'batch_size_train': 4,           # Training batch size (default: 4)
    'batch_size_val': 8,             # Validation batch size (default: 8)
    'lr': 5e-4,                      # Learning rate (lowered to 5e-4 for physics_frno)
    'weight_decay': 1e-4,            # Weight decay (default: 1e-4)
    'patience': 15,                  # Early stopping patience
    'grad_clip': 1.0,                # Gradient clipping norm (default: 1.0)
    'pct_start': 0.15,               # OneCycleLR warmup fraction
    
    # Data Configuration
    'stride_train': 2,               # Sliding window stride for training (default: 2)
    'stride_val': 4,                 # Sliding window stride for validation (default: 4)
    'use_weighted_sampler': True,    # Use weighted random sampling (default: True)
    'month_weights': {               # Month sampling weights (default: baseline)
        'APRIL_16': 0.5,
        'JULY_16': 0.5,
        'OCT_16': 1.5,
        'DEC_16': 3.0               # Winter weighted higher for PM2.5 patterns
    },
    
    # Loss Configuration
    'loss_type': 'log_rmse',        # 'log_rmse', 'weighted_mae', 'three_phase'
    'phase_switch_epoch': 20,        # Epoch to switch from log_rmse to weighted_mae (default: 20)
    'intensity_alpha': 1.5,          # Intensity weighting alpha (default: 1.5)
    'intensity_ref': 59.0,           # Reference concentration for weighting (default: 59.0)
    'intensity_cap': 3.0,            # Max weight multiplier (default: 3.0)
    
    # Horizon Weights (linearly increasing across 16 forecast steps)
    'horizon_weight_min': 0.8,       # Weight for h+1 (default: 0.8)
    'horizon_weight_max': 1.4,       # Weight for h+16 (default: 1.4)
    
    # Checkpoint Configuration
    'checkpoint_metric': 'val_rmse_phys',  # 'val_rmse_std' or 'val_rmse_phys'
    'checkpoint_mixed_alpha': 0.3,   # Alpha for mixed checkpoint metric (0.7*val_obj + 0.3*val_rmse)
    
    # Optimization
    'use_amp': False,                # AMP disabled for TFNO (complex params issue)
    
    # Physics
    'physics_enabled': True,
    'lambda_p': 0.02,
    
    # Inference
    'use_tta': True,                # Test-time augmentation (default: True)
    'use_autoregressive': True,     # Added validation
    'tta_modes': ['identity', 'hflip', 'vflip', 'hvflip'],  # TTA modes
    
    # Persistence Baseline (threshold to beat)
    'persistence_rmse_phys': 30.83, # µg/m³
}

print("="*70)
print("HYPERPARAMETERS LOADED:")
print("="*70)
for key, value in HYPERPARAMS.items():
    print(f"  {key}: {value}")
print("="*70)

# =============================================================================
# CONFIGURATION APPLICATION
# =============================================================================
cfg = load_config()

def apply_tunable_patch(cfg: dict, hp: dict) -> dict:
    """Apply hyperparameter dictionary to config with tunable values."""
    
    # Model configuration
    cfg['model']['type'] = hp['model_type']
    cfg['model']['width'] = hp['width']
    cfg['model']['modes'] = hp['modes']
    cfg['model']['depth'] = hp['depth']
    
    # Feature configuration - only use base features (8 total: cpm25 + 6 met + 1 emis)
    cfg['features']['use_aux'] = hp['use_aux']
    if not hp['use_aux']:
        cfg['features']['aux'] = []
    
    # Base features: cpm25 + met + emis
    cfg['features']['all'] = ['cpm25'] + cfg['features']['met'] + cfg['features']['emis']
    cfg['features']['base'] = list(cfg['features']['all'])
    cfg['features']['input'] = list(cfg['features']['base'])
    cfg['features']['n_features'] = len(cfg['features']['base'])
    cfg['features']['input_channels'] = len(cfg['features']['input'])
    
    # Training configuration
    cfg.setdefault('training', {})
    cfg['training']['epochs'] = hp['epochs']
    cfg['training']['phase1_epochs'] = hp['phase_switch_epoch']
    cfg['training']['phase2_epochs'] = hp['epochs'] - hp['phase_switch_epoch']
    cfg['training']['phase3_epochs'] = 0
    cfg['training']['batch_size_train'] = hp['batch_size_train']
    cfg['training']['batch_size_val'] = hp['batch_size_val']
    cfg['training']['lr'] = hp['lr']
    cfg['training']['weight_decay'] = hp['weight_decay']
    cfg['training']['patience'] = hp['patience']
    cfg['training']['grad_clip'] = hp['grad_clip']
    cfg['training']['pct_start'] = hp['pct_start']
    cfg['training']['stride_train'] = hp['stride_train']
    cfg['training']['stride_val'] = hp['stride_val']
    cfg['training']['use_weighted_sampler'] = hp['use_weighted_sampler']
    cfg['training']['month_sampling_weights'] = hp['month_weights']
    cfg['training']['checkpoint_metric'] = hp['checkpoint_metric']
    cfg['training']['checkpoint_mixed_alpha'] = hp['checkpoint_mixed_alpha']
    cfg['training']['use_amp'] = hp['use_amp']
    cfg['training']['resume_if_available'] = False  # Prevent architecture mismatch crashes
    
    # Loss configuration
    cfg.setdefault('loss', {})
    cfg['loss']['type'] = hp['loss_type']
    cfg['loss']['phase_switch_epoch'] = hp['phase_switch_epoch']
    cfg['loss']['intensity_alpha'] = hp['intensity_alpha']
    cfg['loss']['intensity_ref'] = hp['intensity_ref']
    cfg['loss']['intensity_cap'] = hp['intensity_cap']
    cfg['loss']['horizon_weight_min'] = hp['horizon_weight_min']
    cfg['loss']['horizon_weight_max'] = hp['horizon_weight_max']
    
    # Physics (disable for simple TFNO2D baseline)
    cfg.setdefault('physics', {})
    cfg['physics']['enabled'] = hp.get('physics_enabled', False)
    cfg['training']['lambda_p'] = hp.get('lambda_p', 0.0)
    
    # Inference configuration
    cfg.setdefault('inference', {})
    cfg['inference']['use_tta'] = hp['use_tta']
    cfg['inference']['tta_modes'] = hp['tta_modes']
    cfg['inference']['use_autoregressive'] = hp.get('use_autoregressive', True)
    
    # W&B configuration
    cfg.setdefault('wandb', {})
    cfg['wandb'].setdefault('enabled', True)
    cfg['wandb'].setdefault('run_name', 'exp04-tfno2d-tuned')
    
    return cfg

# Apply hyperparameter patch
cfg = apply_tunable_patch(cfg, HYPERPARAMS)

# Set seeds for reproducibility
seed_everything(cfg['training']['seed'])
print_device_info(cfg['device'])

# Print final configuration summary
print(f"\nBase features ({cfg['features']['n_features']}): {cfg['features']['base']}")
print(f"Aux features  ({len(cfg['features']['aux'])}): {cfg['features']['aux']}")
print(f"Input channels: {cfg['features']['input_channels']}")
print(f"Train months: {cfg['data']['train_months']}")
print(f"Val month:    {cfg['data']['val_month']}")
print(f"Model type:   {cfg['model']['type']}")
print(f"Model config: width={cfg['model']['width']}, modes={cfg['model']['modes']}, depth={cfg['model']['depth']}")
print(f"Normalization:{cfg['preprocessing']['normalization']}")
print(f"AMP enabled:  {cfg['training']['use_amp']}")
print(f"Physics loss: {cfg['physics']['enabled']} | lambda_p={cfg['training']['lambda_p']}")
print(f"Epochs: phase1={cfg['training']['phase1_epochs']} phase2={cfg['training']['phase2_epochs']} phase3={cfg['training']['phase3_epochs']}")
print(f"Batch sizes: train={cfg['training']['batch_size_train']}, val={cfg['training']['batch_size_val']}")
print(f"LR: {cfg['training']['lr']}, Weight decay: {cfg['training']['weight_decay']}")
print(f"Checkpoint metric: {cfg['training']['checkpoint_metric']}")
print(f"W&B: {cfg['wandb']['enabled']} | {cfg['wandb']['entity']}/{cfg['wandb']['project']}")
print(f"Run name: {cfg['wandb']['run_name']}")
print(f"Grid scaler: {cfg['paths']['grid_stats']}")
print(f"Data root:   {cfg['paths']['data']}")

HYPERPARAMETERS LOADED:
  model_type: physics_frno
  width: 64
  modes: 20
  depth: 6
  use_aux: True
  epochs: 50
  batch_size_train: 4
  batch_size_val: 8
  lr: 0.0005
  weight_decay: 0.0001
  patience: 15
  grad_clip: 1.0
  pct_start: 0.15
  stride_train: 2
  stride_val: 4
  use_weighted_sampler: True
  month_weights: {'APRIL_16': 0.5, 'JULY_16': 0.5, 'OCT_16': 1.5, 'DEC_16': 3.0}
  loss_type: log_rmse
  phase_switch_epoch: 20
  intensity_alpha: 1.5
  intensity_ref: 59.0
  intensity_cap: 3.0
  horizon_weight_min: 0.8
  horizon_weight_max: 1.4
  checkpoint_metric: val_rmse_phys
  checkpoint_mixed_alpha: 0.3
  use_amp: False
  physics_enabled: True
  lambda_p: 0.02
  use_tta: True
  use_autoregressive: True
  tta_modes: ['identity', 'hflip', 'vflip', 'hvflip']
  persistence_rmse_phys: 30.83


Device: cuda
GPU: Tesla P100-PCIE-16GB
GPU Memory: 17.1 GB

Base features (10): ['cpm25', 'pblh', 't2', 'u10', 'v10', 'q2', 'rain', 'PM25', 'NOx', 'NH3']
Aux features  (4): ['hour_sin', 'hour_cos', 'rain_mask', 'NMVOC_finn_mask']
Input channels: 10
Train months: ['APRIL_16', 'JULY_16', 'OCT_16']
Val month:    DEC_16
Model type:   physics_frno
Model config: width=64, modes=20, depth=6
Normalization:grid_log_standardize
AMP enabled:  False
Physics loss: True | lambda_p=0.02
Epochs: phase1=20 phase2=30 phase3=0
Batch sizes: train=4, val=8
LR: 0.0005, Weight decay: 0.0001
Checkpoint metric: val_rmse_phys
W&B: True | ronitraj/aisehack
Run name: 
Grid scaler: /kaggle/working/grid_log_scaler_2016.npz
Data root:   /kaggle/input/datasets/khushisingh942004/aisehack


## Raw Data Audit (Unprocessed)

This block inspects the raw `raw/<MONTH>/<feature>.npy` files **before preprocessing**.
It reports per-month/per-feature shape, min, max, mean, std, and finite-ratio using memory-mapped reads with light temporal sampling.

In [4]:
import os
import numpy as np
import pandas as pd

data_root = cfg['paths']['data']
months = list(cfg['data']['months'])
raw_features = list(cfg['features']['all'])

def _light_sample(arr: np.ndarray, max_steps: int = 48) -> np.ndarray:
    step = max(1, int(np.ceil(arr.shape[0] / max_steps)))
    return np.asarray(arr[::step], dtype=np.float32)

raw_rows = []
for month in months:
    month_dir = os.path.join(data_root, 'raw', month)
    time_path = os.path.join(month_dir, 'time.npy')
    t_total = int(np.load(time_path, mmap_mode='r').shape[0]) if os.path.exists(time_path) else None

    for feat in raw_features:
        feat_path = os.path.join(month_dir, f'{feat}.npy')
        arr = np.load(feat_path, mmap_mode='r')  # (T, H, W)
        sample = _light_sample(arr)
        finite_mask = np.isfinite(sample)
        finite_ratio = float(finite_mask.mean())

        raw_rows.append({
            'month': month,
            'feature': feat,
            'shape': str(tuple(arr.shape)),
            'time_steps': t_total if t_total is not None else arr.shape[0],
            'raw_min': float(np.nanmin(sample)),
            'raw_max': float(np.nanmax(sample)),
            'raw_mean': float(np.nanmean(sample)),
            'raw_std': float(np.nanstd(sample)),
            'finite_ratio': finite_ratio,
        })

raw_stats_df = pd.DataFrame(raw_rows)
print(f'Raw audit rows: {len(raw_stats_df)}')
display(raw_stats_df.head(20))

print('\nRaw feature summary across all months:')
summary = (
    raw_stats_df.groupby('feature', as_index=False)
    .agg(
        months=('month', 'nunique'),
        raw_min=('raw_min', 'min'),
        raw_max=('raw_max', 'max'),
        raw_mean=('raw_mean', 'mean'),
        raw_std=('raw_std', 'mean'),
        finite_ratio=('finite_ratio', 'mean'),
    )
    .sort_values('feature')
)
display(summary)

Raw audit rows: 40


,month,feature,shape,time_steps,raw_min,raw_max,raw_mean,raw_std,finite_ratio
0,APRIL_16,cpm25,"(715, 140, 124)",715,0.071024,9.881295e+02,2.237724e+01,2.707841e+01,1.0
1,APRIL_16,pblh,"(715, 140, 124)",715,59.331356,5.806668e+03,9.311025e+02,9.001161e+02,1.0
2,APRIL_16,t2,"(715, 140, 124)",715,237.858582,3.187064e+02,2.930876e+02,1.498711e+01,1.0
3,APRIL_16,u10,"(715, 140, 124)",715,-13.668159,1.812738e+01,2.139477e+00,3.256214e+00,1.0
4,APRIL_16,v10,"(715, 140, 124)",715,-15.714410,2.153572e+01,2.426578e-01,3.135833e+00,1.0
5,APRIL_16,q2,"(715, 140, 124)",715,0.000000,2.796764e-02,1.035440e-02,6.902732e-03,1.0
6,APRIL_16,rain,"(715, 140, 124)",715,0.000000,2.523646e+02,7.124422e-02,1.416599e+00,1.0
7,APRIL_16,PM25,"(715, 140, 124)",715,0.000000,6.268296e-08,5.319164e-11,5.590823e-10,1.0
8,APRIL_16,NOx,"(715, 140, 124)",715,0.000000,3.504497e-08,4.957290e-11,3.646890e-10,1.0
9,APRIL_16,NH3,"(715, 140, 124)",715,0.000000,9.172897e-09,3.456507e-11,9.335507e-11,1.0



Raw feature summary across all months:


,feature,months,raw_min,raw_max,raw_mean,raw_std,finite_ratio
0,NH3,4,0.000000,9.172897e-09,3.017982e-11,5.750718e-11,1.0
1,NOx,4,0.000000,3.504497e-08,4.133049e-11,3.263201e-10,1.0
2,PM25,4,0.000000,6.268296e-08,3.482478e-11,2.652365e-10,1.0
3,cpm25,4,0.001774,2.260008e+03,3.349975e+01,4.385409e+01,1.0
4,pblh,4,54.382618,5.806668e+03,7.540322e+02,6.072295e+02,1.0
5,q2,4,0.000000,3.258686e-02,1.142620e-02,6.513126e-03,1.0
6,rain,4,0.000000,3.970356e+02,2.477296e-01,2.933756e+00,1.0
7,t2,4,227.966003,3.206241e+02,2.914451e+02,1.357352e+01,1.0
8,u10,4,-25.017935,2.586859e+01,1.543503e+00,3.444427e+00,1.0
9,v10,4,-25.034006,2.510244e+01,1.405338e-01,2.879858e+00,1.0


## 2. Load Bounds + Build/Load Grid Scaler (Current Preprocessing)

In [5]:
# Load official bounds (used for diagnostics / inverse transforms)
bounds = load_minmax_bounds(cfg)
sanity_check_bounds(bounds, cfg['features']['all'])

# Kaggle safety: scaler must never be saved under read-only /kaggle/input
if os.path.exists('/kaggle') and str(cfg['paths']['grid_stats']).startswith('/kaggle/input/'):
    cfg['paths']['grid_stats'] = os.path.join('/kaggle/working', os.path.basename(cfg['paths']['grid_stats']))
    print(f"Redirected writable grid scaler path: {cfg['paths']['grid_stats']}")

if not os.path.exists(cfg['paths']['grid_stats']):
    grid_stats = build_grid_stats(cfg, bounds=bounds)
else:
    grid_stats = load_grid_stats(cfg)

cfg.setdefault('_runtime', {})['grid_stats'] = grid_stats
print(f"Loaded grid scaler for {len(grid_stats)} standardized features")
print(f"Source: {cfg['paths']['grid_stats']}")
describe_grid_stats(grid_stats, cfg['features']['input'])


Feature                    Min            Max          Range   OK?
─────────────────────────────────────────────────────────────────
cpm25                 0.993999        1465.25        1464.25  ✓
pblh                   52.1152        6271.37        6219.25  ✓
t2                     223.531        324.774        101.244  ✓
u10                   -26.8288        29.0259        55.8547  ✓
v10                   -29.2164        31.9303        61.1467  ✓
q2                           0      0.0458545      0.0458545  ✓
rain                         0        96.6274        96.6274  ✓
PM25                         0    1.42691e-07    1.42691e-07  ✓
NOx                          0    7.97706e-08    7.97706e-08  ✓
NH3                          0     2.0868e-08     2.0868e-08  ✓

Loaded grid scaler for 10 standardized features
Source: /kaggle/working/grid_log_scaler_2016.npz

Feature                               mean[min,max]                   std[min,max]
────────────────────────────────────────────

## Preprocessed Data Audit (After Pipeline Transforms)

This block applies the current preprocessing (`log1p`, `log(x+eps)`, signed-log + grid standardization)
to sampled raw tensors and reports post-transform statistics for side-by-side inspection.

In [6]:
import os
import numpy as np
import pandas as pd
from src.data import normalize_feature

prep_rows = []
for month in months:
    month_dir = os.path.join(data_root, 'raw', month)
    for feat in raw_features:
        feat_path = os.path.join(month_dir, f'{feat}.npy')
        raw_arr = np.load(feat_path, mmap_mode='r')
        raw_sample = _light_sample(raw_arr)
        pre_sample = normalize_feature(raw_sample, feat, bounds, cfg, grid_stats)

        prep_rows.append({
            'month': month,
            'feature': feat,
            'raw_min': float(np.nanmin(raw_sample)),
            'raw_max': float(np.nanmax(raw_sample)),
            'raw_mean': float(np.nanmean(raw_sample)),
            'raw_std': float(np.nanstd(raw_sample)),
            'prep_min': float(np.nanmin(pre_sample)),
            'prep_max': float(np.nanmax(pre_sample)),
            'prep_mean': float(np.nanmean(pre_sample)),
            'prep_std': float(np.nanstd(pre_sample)),
            'prep_finite_ratio': float(np.isfinite(pre_sample).mean()),
        })

prep_stats_df = pd.DataFrame(prep_rows)
print(f'Preprocessed audit rows: {len(prep_stats_df)}')
display(prep_stats_df.head(20))

print('\nPer-feature comparison (averaged across months):')
prep_summary = (
    prep_stats_df.groupby('feature', as_index=False)
    .agg(
        raw_mean=('raw_mean', 'mean'),
        raw_std=('raw_std', 'mean'),
        prep_mean=('prep_mean', 'mean'),
        prep_std=('prep_std', 'mean'),
        prep_finite_ratio=('prep_finite_ratio', 'mean'),
    )
    .sort_values('feature')
)
display(prep_summary)

print('\nCheck cpm25 post-preprocess distribution by month:')
cpm_month = prep_stats_df[prep_stats_df['feature'] == 'cpm25'][['month', 'prep_min', 'prep_max', 'prep_mean', 'prep_std', 'prep_finite_ratio']]
display(cpm_month.sort_values('month'))

Preprocessed audit rows: 40


,month,feature,raw_min,raw_max,raw_mean,raw_std,prep_min,prep_max,prep_mean,prep_std,prep_finite_ratio
0,APRIL_16,cpm25,0.071024,9.881295e+02,2.237724e+01,2.707841e+01,-4.208161,4.973223,-0.031599,0.889057,1.0
1,APRIL_16,pblh,59.331356,5.806668e+03,9.311025e+02,9.001161e+02,-4.249217,10.705320,0.035451,1.163859,1.0
2,APRIL_16,t2,237.858582,3.187064e+02,2.930876e+02,1.498711e+01,-4.965415,4.166523,0.050751,1.071120,1.0
3,APRIL_16,u10,-13.668159,1.812738e+01,2.139477e+00,3.256214e+00,-4.706103,4.290593,0.067474,0.955660,1.0
4,APRIL_16,v10,-15.714410,2.153572e+01,2.426578e-01,3.135833e+00,-4.461776,4.542158,-0.122727,0.995211,1.0
5,APRIL_16,q2,0.000000,2.796764e-02,1.035440e-02,6.902732e-03,-5.784167,7.311983,-0.530077,0.760942,1.0
6,APRIL_16,rain,0.000000,2.523646e+02,7.124422e-02,1.416599e+00,-0.771827,45.691601,-0.121285,0.838736,1.0
7,APRIL_16,PM25,0.000000,6.268296e-08,5.319164e-11,5.590823e-10,-8.462906,19.664248,0.614988,1.134023,1.0
8,APRIL_16,NOx,0.000000,3.504497e-08,4.957290e-11,3.646890e-10,-5.949020,19.038807,1.202923,1.515729,1.0
9,APRIL_16,NH3,0.000000,9.172897e-09,3.456507e-11,9.335507e-11,-1.199706,18.131699,0.609448,0.813877,1.0



Per-feature comparison (averaged across months):


,feature,raw_mean,raw_std,prep_mean,prep_std,prep_finite_ratio
0,NH3,3.017982e-11,5.750718e-11,0.271400,3.389393,1.0
1,NOx,4.133049e-11,3.263201e-10,0.505062,10.431636,1.0
2,PM25,3.482478e-11,2.652365e-10,3.913928,20.613185,1.0
3,cpm25,3.349975e+01,4.385409e+01,0.152841,1.119706,1.0
4,pblh,7.540322e+02,6.072295e+02,-0.093030,0.995105,1.0
5,q2,1.142620e-02,6.513126e-03,-0.350299,0.799177,1.0
6,rain,2.477296e-01,2.933756e+00,0.102851,3.193471,1.0
7,t2,2.914451e+02,1.357352e+01,-0.444005,1.043934,1.0
8,u10,1.543503e+00,3.444427e+00,-0.205247,1.084391,1.0
9,v10,1.405338e-01,2.879858e+00,-0.138075,1.023499,1.0



Check cpm25 post-preprocess distribution by month:


,month,prep_min,prep_max,prep_mean,prep_std,prep_finite_ratio
0,APRIL_16,-4.208161,4.973223,-0.031599,0.889057,1.0
30,DEC_16,-5.024980,7.775259,0.610551,1.674765,1.0
10,JULY_16,-5.148881,5.672787,-0.420226,0.915033,1.0
20,OCT_16,-3.658220,5.722383,0.452638,0.999971,1.0


## 3. Load & Preprocess Training / Validation Data

Current preprocessing used here comes directly from `src/data.py` + config:
- feature-wise transforms (`log1p`, `log(x+eps)`, signed-log)
- grid-wise scaler normalization (`grid_log_standardize`)
- strict temporal blocking (`OCT_16` held out entirely)
- lazy sliding-window dataset to avoid materializing massive tensors

In [7]:
print("Loading + normalizing training months ...")
train_data = load_all_months(cfg, cfg['data']['train_months'], bounds)

print("\nLoading + normalizing validation month ...")
val_data = load_all_months(cfg, [cfg['data']['val_month']], bounds)

Loading + normalizing training months ...
  Loading APRIL_16 ... OK
  Loading JULY_16 ... OK
  Loading OCT_16 ... OK

Loading + normalizing validation month ...
  Loading DEC_16 ... OK


In [8]:
train_dl, val_dl = get_dataloaders(cfg, train_data, val_data, bounds)
print("Batch shape check ...")
xb, yb = next(iter(train_dl))
print(f"  x: {tuple(xb.shape)}  (B, C={xb.shape[1]}, T={xb.shape[2]}, H={xb.shape[3]}, W={xb.shape[4]})")
print(f"  y: {tuple(yb.shape)}  (B, H, W, T_out={yb.shape[3]})")
print(f"  x range: [{xb.min():.3f}, {xb.max():.3f}]")
print(f"  y range: [{yb.min():.3f}, {yb.max():.3f}]")
feat_to_idx = {name: i for i, name in enumerate(cfg['features']['input'])}
print(f"  PhysicsFrNO mode: aux_enabled={cfg['features']['use_aux']} checkpoint_resume={cfg['training']['resume_if_available']}")
if 'cpm25' in feat_to_idx:
    cpm_idx = feat_to_idx['cpm25']
    tail_energy = xb[:, cpm_idx, cfg['time']['t_in_cpm']:, :, :].abs().mean().item()
    print(f"  cpm25 tail mean abs after t={cfg['time']['t_in_cpm']}: {tail_energy:.6f} (should be ~0)")

  Temporal firewall (hours): 12
  Train skip-start: {}
  Train skip-end:   {'OCT_16': 12}
  Val skip-start:   {'DEC_16': 12}
  Val skip-end:     {}
  Intra-month split on DEC_16: train_head≈80% | val_tail≈20% | split_hour=591
  Train samples: 1,343  |  Val samples: 28
  Computing intensity-based weights for train_ds...
Batch shape check ...
  x: (4, 10, 26, 140, 124)  (B, C=10, T=26, H=140, W=124)
  y: (4, 140, 124, 16)  (B, H, W, T_out=16)
  x range: [-76.130, 9459.621]
  y range: [-4.719, 6.368]
  PhysicsFrNO mode: aux_enabled=True checkpoint_resume=False
  cpm25 tail mean abs after t=10: 0.000000 (should be ~0)


## 4. Build & Train PhysicsFrNO

In [9]:
model = build_model(cfg)
print(f"Parameters: {count_parameters(model):,}")
print(f"Using model: {cfg['model']['type']}")
print(f"Width={cfg['model'].get('width', 'n/a')} Modes={cfg['model'].get('modes', 'n/a')} Depth={cfg['model'].get('depth', 'n/a')}")
print("Target: stable PhysicsFrNO training with calibrated regularization")

Parameters: 981,159
Using model: physics_frno
Width=64 Modes=20 Depth=6
Target: stable PhysicsFrNO training with calibrated regularization


In [10]:
# ── Weights & Biases — initialise run ─────────────────────────────────────────
try:
    import wandb as _wb
    if _wb.run is not None:
        _wb.finish()
except Exception:
    pass

wandb_run = init_wandb(cfg)  # returns None when disabled

In [11]:
history = train(cfg, model, train_dl, val_dl, bounds=bounds, wandb_run=wandb_run)


──────────────────────────────────────────────────────────────────────
Starting Simple 1-Phase Training Baseline
Global Persistence (Physical): 30.83 µg/m³
──────────────────────────────────────────────────────────────────────

Epoch   1/50 | Train RMSE(std): 0.6239 | Val RMSE(std): 0.6086 | Val RMSE(phys): 35.373 µg/m³ | Sel: 35.3726  ← saved
Epoch   2/50 | Train RMSE(std): 0.5588 | Val RMSE(std): 0.5741 | Val RMSE(phys): 35.026 µg/m³ | Sel: 35.0264  ← saved
Epoch   3/50 | Train RMSE(std): 0.5189 | Val RMSE(std): 0.5598 | Val RMSE(phys): 33.394 µg/m³ | Sel: 33.3941  ← saved


KeyboardInterrupt: 

### Training Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.train import PERSISTENCE_RMSE_PHYS

epochs_x = list(range(1, len(history['val_loss']) + 1))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

ax = axes[0]
ax.plot(epochs_x, history['train_loss'], label='Train RMSE (std-space)', alpha=0.8)
ax.plot(epochs_x, history['val_loss'], label='Val RMSE (std-space)', alpha=0.9)
ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
ax.set_title('Standardized-Space RMSE')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
if 'val_phys_rmse' in history:
    ax.plot(epochs_x, history['val_phys_rmse'], label='Val RMSE (µg/m³)', alpha=0.9)
if 'val_persistence_phys' in history:
    ax.plot(epochs_x, history['val_persistence_phys'], label='Val persistence (µg/m³)', alpha=0.8)
ax.axhline(PERSISTENCE_RMSE_PHYS, color='red', linestyle='--', label=f'Global persistence ({PERSISTENCE_RMSE_PHYS:.2f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE (µg/m³)')
ax.set_title('Physical-Space Validation RMSE')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(epochs_x, history.get('train_objective', history['train_loss']), label='Train objective', alpha=0.8)
ax.plot(epochs_x, history.get('val_objective', history['val_loss']), label='Val objective', alpha=0.9)
ax.plot(epochs_x, history.get('selection_metric', history['val_loss']), label='Selection metric', alpha=0.9)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Objective / Selection')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/images/training_curves_fno.png', dpi=120, bbox_inches='tight')
plt.show()

best_sel = float(np.min(history.get('selection_metric', history['val_loss'])))
best_idx = int(np.argmin(history.get('selection_metric', history['val_loss'])))
best_phys = history.get('val_phys_rmse', [None])[best_idx]

print(f"\n{'─'*55}")
print(f"  Best selection metric: {best_sel:.6f} (epoch {best_idx + 1})")
if best_phys is not None:
    ratio = best_phys / PERSISTENCE_RMSE_PHYS
    print(f"  Best val RMSE (phys):  {best_phys:.3f} µg/m³")
    print(f"  Persistence ratio:     {ratio:.3f}  {'✓ below persistence' if ratio < 1.0 else '✗ above persistence'}")
print(f"{'─'*55}")

## 5. Inference & Submit

In [ ]:
import torch

state = torch.load(cfg['paths']['model_save'], map_location=cfg['device'])
if isinstance(state, dict) and 'model_state_dict' in state:
    model.load_state_dict(state['model_state_dict'])
else:
    model.load_state_dict(state)

preds = run_inference(cfg, model, bounds)
print(f"Predictions saved at: {cfg['paths']['output']}")
print('Done!')

In [ ]:
import os
import numpy as np

# ------------------------------
# Environment-aware path helpers
# ------------------------------
def first_existing(paths):
    return next((p for p in paths if p and os.path.exists(p)), None)

on_kaggle = os.path.exists('/kaggle')

# Resolve source/project root
if 'SRC_ROOT' in globals() and isinstance(SRC_ROOT, str):
    src_root = SRC_ROOT
else:
    cwd = os.getcwd()
    candidate_roots = [
        os.path.abspath(os.path.join(cwd, '..')),
        os.path.abspath(os.path.join(cwd, '../..')),
        os.path.abspath(cwd),
        os.path.abspath('/home/raj/Documents/CODING/Hackathon/ANRF_AISEHack_Code/Ronit'),
        '/kaggle/input/ronit-pm25-src',
    ]
    src_root = first_existing([r for r in candidate_roots if os.path.exists(os.path.join(r, 'outputs'))]) or candidate_roots[0]

# Resolve data root (works for local + Kaggle competition/input variants)
if 'cfg' in globals() and isinstance(cfg, dict) and 'paths' in cfg:
    cfg_output = cfg['paths'].get('output')
    cfg_model_save = cfg['paths'].get('model_save', '')
    cfg_temp = cfg['paths'].get('temp', '')
    data_root = cfg['paths'].get('data')
else:
    cfg_output = None
    cfg_model_save = ''
    cfg_temp = ''
    data_root = None

data_candidates = [
    data_root,
    os.environ.get('AISEHACK_DATA'),
    '/kaggle/input/aisehack-theme-2',
    '/kaggle/input/competitions/aisehack-theme-2',
    '/kaggle/input/datasets/khushisingh942004/aisehack',
    os.path.abspath(os.path.join(src_root, '..', 'aisehack-theme-2')),
    os.path.abspath(os.path.join(src_root, 'aisehack-theme-2')),
]
data_root = first_existing([p for p in data_candidates if p and os.path.exists(os.path.join(p, 'test_in'))])
if data_root is None:
    raise FileNotFoundError('Could not locate data root containing test_in/. Set AISEHACK_DATA or attach competition data.')

# ------------------------------
# Locate preds.npy
# ------------------------------
pred_candidates = [
    '/kaggle/working/preds.npy' if on_kaggle else None,
    cfg_output,
    os.path.join(os.path.dirname(cfg_model_save), 'preds.npy') if cfg_model_save else None,
    os.path.join(os.path.dirname(cfg_temp), 'preds.npy') if cfg_temp else None,
    os.path.abspath(os.path.join(src_root, 'outputs', 'models', 'preds.npy')),
    os.path.abspath(os.path.join(src_root, 'outputs', 'submissions', 'preds.npy')),
]
preds_path = first_existing(pred_candidates)
if preds_path is None:
    raise FileNotFoundError('Could not find preds.npy in expected locations (including /kaggle/working).')

preds = np.load(preds_path)
print(f"Using preds: {preds_path}")
print(f"preds shape (raw): {preds.shape}")

# ------------------------------
# Load available test history
# ------------------------------
test_cpm25_path = os.path.join(data_root, 'test_in', 'cpm25.npy')
test_cpm25_hist = np.load(test_cpm25_path)
print(f"Using test history: {test_cpm25_path}")
print(f"test_in/cpm25 shape: {test_cpm25_hist.shape}")

# Expected:
# test_cpm25_hist: (N, 10, H, W)
# preds:           (N, H, W, 16)

# Handle common prediction layout variant: (N, 16, H, W)
if preds.ndim == 4 and preds.shape[1] == 16 and preds.shape[-1] != 16:
    preds = np.transpose(preds, (0, 2, 3, 1))
    print(f"preds shape (transposed to NHWT): {preds.shape}")

if preds.ndim != 4 or preds.shape[-1] != 16:
    raise ValueError(f"Expected preds shape like (N, H, W, 16), got {preds.shape}")

n, h, w, t_out = preds.shape
if t_out != 16:
    raise ValueError(f"Expected 16 forecast steps, got {t_out}")
if test_cpm25_hist.shape[0] != n or test_cpm25_hist.shape[2] != h or test_cpm25_hist.shape[3] != w:
    raise ValueError(
        'Spatial/sample mismatch between preds and test history: '
        f'preds={preds.shape}, test_hist={test_cpm25_hist.shape}'
    )

last_obs = test_cpm25_hist[:, -1, :, :]  # (N, H, W)

# 1) Horizon-1 consistency with latest observation
h1_pred = preds[..., 0]
rmse_h1_vs_last = float(np.sqrt(np.mean((h1_pred - last_obs) ** 2)))
mae_h1_vs_last = float(np.mean(np.abs(h1_pred - last_obs)))

# 2) Distance from persistence baseline over all forecast horizons
persistence = np.repeat(last_obs[..., None], preds.shape[-1], axis=-1)
rmse_all_vs_persistence = float(np.sqrt(np.mean((preds - persistence) ** 2)))
mae_all_vs_persistence = float(np.mean(np.abs(preds - persistence)))

# 3) Temporal smoothness inside forecast trajectory
step_delta = np.diff(preds, axis=-1)
mean_abs_step_change = float(np.mean(np.abs(step_delta)))

print('\nProxy scores (lower is generally better for RMSE/MAE):')
print(f"- RMSE(H+1 vs last observed):        {rmse_h1_vs_last:.4f}")
print(f"- MAE(H+1 vs last observed):         {mae_h1_vs_last:.4f}")
print(f"- RMSE(all horizons vs persistence): {rmse_all_vs_persistence:.4f}")
print(f"- MAE(all horizons vs persistence):  {mae_all_vs_persistence:.4f}")
print(f"- Mean abs step change (H to H+1):   {mean_abs_step_change:.4f}")

print('\nNote: Official competition score is computed on hidden future targets on Kaggle.')

In [ ]:
# ── Log post-inference proxy scores to W&B then finish run ────────────────────
# These scalars let you compare across runs on the W&B dashboard without
# needing the hidden leaderboard — a lower proxy_rmse_h1 typically tracks well.

try:
    import wandb as _wb
    if wandb_run is not None:
        wandb_run.log({
            'inference/proxy_rmse_h1_vs_last':     rmse_h1_vs_last,
            'inference/proxy_mae_h1_vs_last':      mae_h1_vs_last,
            'inference/proxy_rmse_all_vs_persist': rmse_all_vs_persistence,
            'inference/proxy_mae_all_vs_persist':  mae_all_vs_persistence,
            'inference/mean_abs_step_change':      mean_abs_step_change,
        })
        print("Proxy scores logged to W&B.")
except Exception as _e:
    print(f"Proxy score W&B log skipped: {_e}")

# Finish the W&B run (uploads remaining data, closes the run)
finish_wandb(wandb_run, history, bounds, cfg)